# Pandas · Análisis de ficheros

Leer ficheros CSV con pandas, filtrar, agrupar y reestructurar datos reales. Los ficheros se descargan de los repositorios de [aprendeconalf.es](https://aprendeconalf.es/docencia/python/ejercicios/) y de *30 Days Of Python*.

## 1. Resumen de cotizaciones

Construir un DataFrame a partir de `cotizacion.csv` (separado por `;`, con decimales con coma y miles con punto) y devolver otro con el mínimo, el máximo y la media de cada columna.

In [1]:
import pandas as pd

URL_ALF = "https://raw.githubusercontent.com/asalber/aprendeconalf/master/content/es/docencia/python/ejercicios/soluciones/pandas/"

def resumen_cotizaciones(fichero):
    df = pd.read_csv(fichero, sep=";", decimal=",", thousands=".", index_col=0)
    return pd.DataFrame({"Mínimo": df.min(), "Máximo": df.max(), "Media": df.mean()})

print(resumen_cotizaciones(URL_ALF + "cotizacion.csv"))

             Mínimo       Máximo         Media
Final        1.0165     19705.00  2.796769e+03
Máximo       4.0675     19875.00  3.170113e+03
Mínimo       1.0165     19675.00  3.136510e+03
Volumen   1221.0000  36129692.00  4.252279e+06
Efectivo  2343.0900    145765.44  3.176778e+04


## 2. Pasajeros del Titanic

Con `titanic.csv`: dimensiones, número de datos, nombres de filas y columnas, tipos, primeras y últimas 10 filas; el pasajero 148; las filas pares; los nombres de primera clase ordenados; el porcentaje de supervivientes y fallecidos, en total y por clase; eliminar los pasajeros sin edad; la edad media de las mujeres por clase; añadir una columna booleana de menores de edad; y el porcentaje de supervivientes menores y mayores de edad por clase.

In [2]:
titanic = pd.read_csv(URL_ALF + "titanic.csv", index_col=0)
print(titanic.shape, titanic.size, list(titanic.columns), titanic.index[:5].tolist())
print(titanic.dtypes)
print(titanic.head(10)[["Name", "Sex", "Age"]], titanic.tail(10)[["Name", "Sex", "Age"]], sep="\n")
print(titanic.loc[148])
print(titanic.iloc[::2].shape)
print(titanic[titanic["Pclass"] == 1]["Name"].sort_values().head())
print(titanic["Survived"].value_counts(normalize=True) * 100)
print(titanic.groupby("Pclass")["Survived"].mean() * 100)
titanic = titanic.dropna(subset=["Age"])
print(titanic[titanic["Sex"] == "female"].groupby("Pclass")["Age"].mean())
titanic["Menor"] = titanic["Age"] < 18
print(titanic.groupby(["Pclass", "Menor"])["Survived"].mean().unstack() * 100)

(891, 11) 9801 ['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'] [1, 2, 3, 4, 5]
Survived      int64
Pclass        int64
Name            str
Sex             str
Age         float64
SibSp         int64
Parch         int64
Ticket          str
Fare        float64
Cabin           str
Embarked        str
dtype: object
                                                          Name     Sex   Age
PassengerId                                                                 
1                                      Braund, Mr. Owen Harris    male  22.0
2            Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0
3                                       Heikkinen, Miss. Laina  female  26.0
4                 Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0
5                                     Allen, Mr. William Henry    male  35.0
6                                             Moran, Mr. James    male   NaN
7                

## 3. Emisiones contaminantes de Madrid

Con los ficheros `emisiones-2016.csv` a `emisiones-2019.csv`: unirlos; quedarse con estación, magnitud, año, mes y los días; pasar los días a una única columna; crear la fecha y eliminar las no válidas; ordenar por estación, contaminante y fecha; y mostrar estaciones y contaminantes disponibles.

In [3]:
import numpy as np

emisiones = pd.concat([pd.read_csv(URL_ALF + f"emisiones-{año}.csv", sep=";") for año in range(2016, 2020)])
dias = [c for c in emisiones.columns if c.startswith("D")]
emisiones = emisiones[["ESTACION", "MAGNITUD", "ANO", "MES"] + dias]
emisiones = emisiones.melt(id_vars=["ESTACION", "MAGNITUD", "ANO", "MES"], var_name="DIA", value_name="VALOR")
emisiones["DIA"] = emisiones["DIA"].str[1:]
# errors="coerce" convierte fechas imposibles (31 de abril) en NaT
emisiones["FECHA"] = pd.to_datetime(emisiones["ANO"].astype(str) + "-" + emisiones["MES"].astype(str) + "-" + emisiones["DIA"], errors="coerce")
emisiones = emisiones[~np.isnat(emisiones["FECHA"].to_numpy())].sort_values(["ESTACION", "MAGNITUD", "FECHA"])
print(emisiones.shape)
print(sorted(emisiones["ESTACION"].unique().tolist()), sorted(emisiones["MAGNITUD"].unique().tolist()))

(221158, 7)
[4, 8, 11, 16, 17, 18, 24, 27, 35, 36, 38, 39, 40, 47, 48, 49, 50, 54, 55, 56, 57, 58, 59, 60] [1, 6, 7, 8, 9, 10, 12, 14, 20, 30, 35, 42, 43, 44]


## 4. Consultas sobre las emisiones

Funciones para: la serie de un contaminante en una estación y rango de fechas; resúmenes descriptivos por contaminante y por estación (cada estación está en un distrito); el resumen de un contaminante en una estación; las medias mensuales de un contaminante en un año para todas las estaciones; y las medias mensuales de cada contaminante en una estación.

In [4]:
def evolucion(estacion, contaminante, desde, hasta):
    filtro = (emisiones["ESTACION"] == estacion) & (emisiones["MAGNITUD"] == contaminante) & emisiones["FECHA"].between(desde, hasta)
    return emisiones[filtro].set_index("FECHA")["VALOR"]

def resumen(estacion, contaminante):
    return emisiones[(emisiones["ESTACION"] == estacion) & (emisiones["MAGNITUD"] == contaminante)]["VALOR"].describe()

def medias_mensuales(contaminante, año):
    datos = emisiones[(emisiones["MAGNITUD"] == contaminante) & (emisiones["ANO"] == año)]
    return datos.groupby(["ESTACION", "MES"])["VALOR"].mean().unstack()

def medias_por_contaminante(estacion):
    return emisiones[emisiones["ESTACION"] == estacion].groupby(["MAGNITUD", "MES"])["VALOR"].mean().unstack()

print(evolucion(4, 1, "2016-01-01", "2016-01-07"))
print(emisiones.groupby("MAGNITUD")["VALOR"].describe().head())
print(emisiones.groupby(["ESTACION", "MAGNITUD"])["VALOR"].describe().head())
print(resumen(4, 8))
print(medias_mensuales(8, 2019).round(1).iloc[:3, :6])
print(medias_por_contaminante(4).round(1).iloc[:3, :6])

FECHA
2016-01-01    8.0
2016-01-02    7.0
2016-01-03    6.0
2016-01-04    6.0
2016-01-05    7.0
2016-01-06    6.0
2016-01-07    6.0
Name: VALOR, dtype: float64
            count       mean         std  min   25%   50%   75%      max
MAGNITUD                                                                
1         14610.0   7.428953    7.012504  0.0   4.0   7.0  10.0    610.0
6         14610.0   0.350233    0.215935  0.0   0.2   0.3   0.4     14.9
7         35064.0  20.446412  135.123509  0.0   4.0   9.0  23.0  24742.0
8         35064.0  37.677618   20.118050  0.0  22.0  35.0  50.0    148.0
9          8948.0  10.087729   10.643591  0.0   6.0   9.0  13.0    850.0
                    count       mean        std  min   25%   50%    75%    max
ESTACION MAGNITUD                                                             
4        1         1461.0   7.329911  16.379050  1.0   4.0   7.0    9.0  610.0
         6         1461.0   0.411499   0.172902  0.1   0.3   0.4    0.5    1.3
         7   

## 5. Hacker News

Leer `hacker_news.csv`: primeras y últimas cinco filas, la columna de títulos como serie, número de filas y columnas, títulos que contienen Python y JavaScript, y explorar los datos (los más votados y comentados).

In [5]:
hn = pd.read_csv("https://raw.githubusercontent.com/Asabeneh/30-Days-Of-Python/master/data/hacker_news.csv")
print(hn.head()[["title", "num_points"]], hn.tail()[["title", "num_points"]], sep="\n")
titulos = hn["title"]
print(type(titulos), hn.shape)
print(titulos[titulos.str.contains("python", case=False)].head(3).tolist())
print(titulos[titulos.str.contains("javascript", case=False)].head(3).tolist())
print(hn.sort_values("num_points", ascending=False)[["title", "num_points", "num_comments"]].head())
print(hn[["num_points", "num_comments"]].describe())

                                               title  num_points
0                          Interactive Dynamic Video         386
1  Florida DJs May Face Felony for April Fools' W...           2
2       Technology ventures: From Idea to Enterprise           3
3  Note by Note: The Making of Steinway L1037 (2007)           8
4  Title II kills investment? Comcast and other I...          53
                                                   title  num_points
20094  How Purism Avoids Intels Active Management Tec...          10
20095          YC Application Translated and Broken Down           4
20096  Microkernels are slow and Elvis didn't do no d...         169
20097                      How Product Hunt really works         695
20098  RoboBrowser: Your friendly neighborhood web sc...         182
<class 'pandas.Series'> (20099, 7)
['From Python to Lua: Why We Switched', 'Ubuntu 16.04 LTS to Ship Without Python 2', 'Create a GUI Application Using Qt and Python in Minutes']
['Show HN: Hire J